# 02 Feature engineering: RM-relevant features

In [1]:
import sys
from pathlib import Path
ROOT = next(
    (parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (parent / 'src' / 'feature_engineering.py').is_file()),
    Path.cwd().resolve(),
)
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 50)
from feature_engineering import load_raw, engineer_features

df = engineer_features(load_raw())
for c in ['segment', 'life_stage', 'risk_profile']:
    print(df[c].value_counts(), '\\n')

segment
Mass                 24716
Emerging Affluent    12752
Affluent              5348
HNI                   2395
Name: count, dtype: int64 \n
life_stage
Mid Career        19049
Early Career      15571
Pre-Retirement     8067
Young Adult        1336
Senior             1188
Name: count, dtype: int64 \n
risk_profile
Moderate        29224
Conservative    10399
Aggressive       5588
Name: count, dtype: int64 \n


In [2]:
def conv(col):
    return df.groupby(col, observed=True)['y'].apply(lambda s: (s == 'yes').mean() * 100).round(1)

print(conv('segment'))
print(conv('risk_profile'))
print(conv('life_stage'))

segment
Affluent             17.5
Emerging Affluent    13.3
HNI                  15.0
Mass                  9.3
Name: y, dtype: float64
risk_profile
Aggressive      17.3
Conservative    12.2
Moderate        10.5
Name: y, dtype: float64
life_stage
Early Career      12.0
Mid Career         9.4
Pre-Retirement    10.1
Senior            42.3
Young Adult       24.0
Name: y, dtype: float64


In [3]:
df['score_band'] = pd.qcut(df['investment_score'], 5, duplicates='drop')
print(conv('score_band'))

score_band
(-0.001, 30.0]     6.2
(30.0, 36.0]       7.4
(36.0, 60.0]       9.1
(60.0, 65.0]      18.3
(65.0, 100.0]     20.5
Name: y, dtype: float64


In [4]:
from src.recommender import recommend
keys = ['age', 'job', 'marital', 'education', 'balance', 'housing', 'loan', 'default']
sample = df.sample(300, random_state=1)
prim = [recommend(r)['primary'].split(' — ')[0] for r in sample[keys].to_dict('records')]
pd.Series(prim).value_counts()

Term Life Insurance               220
Regular Savings Account            34
NPS (National Pension System)      18
Privilege Banking                  17
iWish Goal-Based RD                 9
Senior Citizens Savings Scheme      2
Name: count, dtype: int64

In [5]:
df.drop(columns=['score_band']).to_csv(ROOT / 'data' / 'processed' / 'customer_profiles.csv', index=False)
print('saved')

saved
